<a href="https://colab.research.google.com/github/vijayanagarcoding/AI-Lab-Colab-book/blob/main/AI_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import random

class VacuumEnvironment:
    def __init__(self):

        self.location_state = {
            'A': random.choice(['Clean', 'Dirty']),
            'B': random.choice(['Clean', 'Dirty'])
        }

        self.vacuum_location = random.choice(['A', 'B'])

    def perceive(self):

        current_state = self.location_state[self.vacuum_location]
        return self.vacuum_location, current_state

    def execute_action(self, action):

        if action == 'Suck':
            self.location_state[self.vacuum_location] = 'Clean'
            print(f"Action: [Suck] - Cleaned Room {self.vacuum_location}")
        elif action == 'Right':
            self.vacuum_location = 'B'
            print("Action: [Move Right] - Moved to Room B")
        elif action == 'Left':
            self.vacuum_location = 'A'
            print("Action: [Move Left] - Moved to Room A")
        elif action == 'NoOp':
            print("Action: [NoOp] - Both rooms are clean. Idle.")


class ReflexVacuumAgent:
    def decide_action(self, percept):

        location, state = percept

        if state == 'Dirty':
            return 'Suck'
        elif location == 'A':
            return 'Right'
        elif location == 'B':
            return 'Left'


def run_simulation(steps=5):
    env = VacuumEnvironment()
    agent = ReflexVacuumAgent()

    print("=== INITIAL ENVIRONMENT STATE ===")
    print(f"Vacuum Location : Room {env.vacuum_location}")
    print(f"Room A State    : {env.location_state['A']}")
    print(f"Room B State    : {env.location_state['B']}")
    print("=" * 32 + "\n")

    for step in range(1, steps + 1):
        # 1. Vacuum perceives current room and its state
        percept = env.perceive()
        print(f"Step {step} | Location: Room {percept[0]} | State: {percept[1]}")

        # 2. Check if whole environment is already clean
        if env.location_state['A'] == 'Clean' and env.location_state['B'] == 'Clean' and percept[1] == 'Clean':
            env.execute_action('NoOp')
            print("\nEnvironment fully cleaned!")
            break

        # 3. Agent decides and executes action
        action = agent.decide_action(percept)
        env.execute_action(action)
        print("-" * 32)

if __name__ == "__main__":
    run_simulation()

=== INITIAL ENVIRONMENT STATE ===
Vacuum Location : Room A
Room A State    : Clean
Room B State    : Dirty

Step 1 | Location: Room A | State: Clean
Action: [Move Right] - Moved to Room B
--------------------------------
Step 2 | Location: Room B | State: Dirty
Action: [Suck] - Cleaned Room B
--------------------------------
Step 3 | Location: Room B | State: Clean
Action: [NoOp] - Both rooms are clean. Idle.

Environment fully cleaned!


In [4]:
import math


def print_board(board):
    print("\n")
    print(f" {board[0]} | {board[1]} | {board[2]} ")
    print("---|---|---")
    print(f" {board[3]} | {board[4]} | {board[5]} ")
    print("---|---|---")
    print(f" {board[6]} | {board[7]} | {board[8]} ")
    print("\n")


def check_win(board, player):

    win_conditions = [
        # Horizontal
        (0, 1, 2), (3, 4, 5), (6, 7, 8),
        # Vertical
        (0, 3, 6), (1, 4, 7), (2, 5, 8),
        # Diagonal
        (0, 4, 8), (2, 4, 6)
    ]
    for a, b, c in win_conditions:
        if board[a] == board[b] == board[c] == player:
            return True
    return False


def check_draw(board):

    return all(cell in ['X', 'O'] for cell in board)


def get_available_moves(board):

    return [i for i, cell in enumerate(board) if cell not in ['X', 'O']]


def minimax(board, depth, is_maximizing):

    if check_win(board, 'O'):
        return 10 - depth
    if check_win(board, 'X'):
        return depth - 10
    if check_draw(board):
        return 0

    if is_maximizing:
        best_score = -math.inf
        for move in get_available_moves(board):
            board[move] = 'O'
            score = minimax(board, depth + 1, False)
            board[move] = str(move + 1)  # Undo move
            best_score = max(score, best_score)
        return best_score
    else:
        best_score = math.inf
        for move in get_available_moves(board):
            board[move] = 'X'
            score = minimax(board, depth + 1, True)
            board[move] = str(move + 1)  # Undo move
            best_score = min(score, best_score)
        return best_score


def get_computer_move(board):
    best_score = -math.inf
    best_move = None

    for move in get_available_moves(board):
        board[move] = 'O'
        score = minimax(board, 0, False)
        board[move] = str(move + 1)  # Undo move

        if score > best_score:
            best_score = score
            best_move = move

    return best_move


def get_human_move(board):

    while True:
        try:
            move = int(input("Your turn (X), enter position (1-9): ")) - 1
            if move < 0 or move > 8:
                print("Invalid input. Choose a number between 1 and 9.")
            elif board[move] in ['X', 'O']:
                print("Spot already taken! Choose another spot.")
            else:
                return move
        except ValueError:
            print("Invalid input. Enter a valid number (1-9).")


def play_game():

    board = [str(i + 1) for i in range(9)]
    current_player = 'X'  # Human starts first

    print("=== Tic-Tac-Toe: You (X) vs Computer (O) ===")
    print_board(board)

    while True:
        if current_player == 'X':
            move = get_human_move(board)
            board[move] = 'X'
        else:
            print("Computer (O) is thinking...")
            move = get_computer_move(board)
            board[move] = 'O'
            print(f"Computer placed 'O' at position {move + 1}")

        print_board(board)

        # Check win condition
        if check_win(board, current_player):
            if current_player == 'X':
                print(" You won!")
            else:
                print(" Computer (O) wins!")
            break

        # Check draw condition
        if check_draw(board):
            print(" It's a draw!")
            break

        # Switch turns
        current_player = 'O' if current_player == 'X' else 'X'


if __name__ == "__main__":
    play_game()

=== Tic-Tac-Toe: You (X) vs Computer (O) ===


 1 | 2 | 3 
---|---|---
 4 | 5 | 6 
---|---|---
 7 | 8 | 9 


Your turn (X), enter position (1-9): 7


 1 | 2 | 3 
---|---|---
 4 | 5 | 6 
---|---|---
 X | 8 | 9 


Computer (O) is thinking...
Computer placed 'O' at position 5


 1 | 2 | 3 
---|---|---
 4 | O | 6 
---|---|---
 X | 8 | 9 


Your turn (X), enter position (1-9): 9


 1 | 2 | 3 
---|---|---
 4 | O | 6 
---|---|---
 X | 8 | X 


Computer (O) is thinking...
Computer placed 'O' at position 8


 1 | 2 | 3 
---|---|---
 4 | O | 6 
---|---|---
 X | O | X 


Your turn (X), enter position (1-9): 2


 1 | X | 3 
---|---|---
 4 | O | 6 
---|---|---
 X | O | X 


Computer (O) is thinking...
Computer placed 'O' at position 1


 O | X | 3 
---|---|---
 4 | O | 6 
---|---|---
 X | O | X 


Your turn (X), enter position (1-9): 4


 O | X | 3 
---|---|---
 X | O | 6 
---|---|---
 X | O | X 


Computer (O) is thinking...
Computer placed 'O' at position 3


 O | X | O 
---|---|---
 X | O | 6 
-